# Pre-Event Survey Data Cleaning

This notebook prepares the student and teacher survey responses for
analysis.

The process includes loading the original files, simplifying column
names, cleaning text, combining school information, checking possible
duplicates, and preparing readable Excel outputs.

Original files remain in `data/raw/`. Cleaned workbooks are saved in
`data/cleaned/`.

Possible duplicates and contradictory answers are flagged for review
rather than automatically removed or corrected.

## 1. Install Required Packages

Install pandas for data processing and openpyxl for creating formatted
Excel workbooks.

Run this cell if the packages are not installed in the selected
notebook environment.

In [1]:
%pip install pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


## 2. Import Libraries and Locate the Project

Import the tools needed for cleaning, validation, and Excel formatting.

Locate the project folder containing `data/raw/`, then create the
`data/cleaned/` folder if it does not already exist.

In [2]:
from pathlib import Path
from datetime import datetime
from math import ceil
import re

import pandas as pd

from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter


current_folder = Path.cwd().resolve()

ROOT = next(
    (
        folder
        for folder in [current_folder, *current_folder.parents]
        if (folder / "data" / "raw").is_dir()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "Cannot find data/raw. Open the notebook inside your project."
    )

RAW = ROOT / "data" / "raw"
CLEANED = ROOT / "data" / "cleaned"

CLEANED.mkdir(parents=True, exist_ok=True)

print("Project folder:", ROOT)
print("Input folder:", RAW)
print("Output folder:", CLEANED)

Project folder: C:\Users\User\Desktop\DEVCON-DATA-ANALYTICS\SEPTEMBER _6_DATA
Input folder: C:\Users\User\Desktop\DEVCON-DATA-ANALYTICS\SEPTEMBER _6_DATA\data\raw
Output folder: C:\Users\User\Desktop\DEVCON-DATA-ANALYTICS\SEPTEMBER _6_DATA\data\cleaned


## 3. Load the Original Survey Responses

Load only the `Form Responses 1` sheet from each workbook.

Other sheets are not appended because they may contain overlapping or
edited responses.

Explicit text answers such as "None" and "NA" are preserved. Empty
responses will be handled during cleaning.

In [3]:
students_raw = pd.read_excel(
    RAW / "students_pre_event.xlsx",
    sheet_name="Form Responses 1",
    keep_default_na=False,
)

teachers_raw = pd.read_excel(
    RAW / "teachers_pre_event.xlsx",
    sheet_name="Form Responses 1",
    keep_default_na=False,
)

print("Students:", students_raw.shape)
print("Teachers:", teachers_raw.shape)

display(students_raw.head(3))
display(teachers_raw.head(3))

Students: (241, 42)
Teachers: (55, 57)


,Timestamp,Grade Level,First Name,Last Name,Gender,Age,Have you attended a DEVCON Kids event before?,How familiar are you with climate change?,Which of the following are ways technology can help address climate change?,"Before today, have you heard of ""Computational Thinking""?",...,I can create step-by-step instructions to solve a problem.,I can improve my ideas after testing them.,I can use computational thinking to solve real-world problems.,I believe my ideas can help improve my community.,I can work with others to solve environmental problems.,Young people can make a difference in protecting the environment.,I feel capable of creating technology that helps others.,I see myself as someone who can create technology.,Column 35,School Name 3
0,2026-08-11 22:46:33.720,Grade 5,KRIEZYLE JOY,CATIBAN,Female,11,No,Somewhat familiar,Track environmental data,No,...,3,4,3,5,5,5,5,5,,
1,2026-08-12 15:50:02.757,Grade 5,Dexter,Tamma,Male,11,No,Somewhat familiar,"Monitor weather and floods, Track environmenta...",No,...,5,5,5,5,5,5,5,4,,
2,2026-08-12 19:45:01.291,Grade 5,Maria Loucyta,Montes,Female,10,No,Heard about it,Reduce waste through smart solutions,I'm not sure,...,4,4,4,4,4,4,5,4,,


,Timestamp,Full Name,Contact Number,Column 3,Which day(s) will you attend the Climate Champions Tech Summit?,Position,Grade Level(s) You Teach,School Cluster,School Name,School Cluster 2,...,How often do you .... Mark only one oval per row. [Use PBL],How often do you .... Mark only one oval per row. [Connect technology to real-world problems],"What challenges do you face when teaching computational thinking, coding, STEM, or climate-related topics? (Check all that apply)",How familiar are you with climate change and climate resilience?,What local issues concern your students?,Which of the following local environmental challenges do you believe technology could help address?,What do you hope to gain from this professional learning program?,What is your biggest concern about teaching computational thinking or coding?,Do you have any accessibility or accommodation needs that we should be aware of to help you participate fully in this event? (Optional),"Your privacy is important to us. The information you provide in this survey will be used only for participant registration, event planning, communication, and evaluating the learning outcomes of the Climate Champions Tech Summit. Survey responses may also be used in summary reports to help improve future programs. Personal information will be kept confidential and processed in accordance with the Data Privacy Act of 2012 (Republic Act No. 10173)."
0,2026-08-07 19:41:09.597,,,,Both Day 1 and Day 2,Teacher,Not applicable,,,,...,Rarely,Rarely,Limited devices,Somewhat familiar,Flooding,Flooding,more knowledge that can be used in classroom s...,If thers device available,,I have read and understood the Data Privacy No...
1,2026-08-10 11:07:50.399,,,,Both Day 1 and Day 2,Teacher,Grade 8,,,2,...,Rarely,Often,"Limited devices, Lack of training, Lack of cla...",Very familiar,"Flooding, Typhoons, Extreme heat, Water shorta...","Flooding, Typhoons, Waste management, Plastic ...","Training, new strategies to help me in my teac...",If i can memorize and do coding by \nmyself,None,I have read and understood the Data Privacy No...
2,2026-08-11 13:51:59.056,,,,Both Day 1 and Day 2,Teacher,Grade 7,,,10,...,Never,Often,"Limited devices, Limited internet, Lack of tra...",Very familiar,"Flooding, Typhoons, Extreme heat, Air pollution","Flooding, Typhoons, Extreme heat, Water shorta...",I hope to gain new knowledge and practical ski...,My biggest concern is understanding coding and...,,I have read and understood the Data Privacy No...


## 4. Assign Shorter Column Names

Replace long survey-question headers with descriptive names that are
easier to use in Python.

The name lists follow the column order in the supplied source files.
Check the column counts and inspect the mapping before continuing.

Preserve the original questions for the exported Dictionary sheet.

In [4]:
STUDENT_COLUMNS = """
timestamp grade_level first_name last_name gender age
prior_devcon climate_familiarity climate_tech_uses heard_of_ct
technologies_used project_experience tech_confidence
environment_belief tech_interest community_tech_interest
environmental_challenges climate_problem learning_goals
accommodation_needs privacy_consent stem_interest attendance_days
cluster_a school_a cluster_b school_b empty_ct ct_familiarity
ct_definition ct_decomposition ct_patterns ct_steps ct_iteration
ct_application community_ideas environmental_teamwork youth_impact
tech_creation_confidence tech_creator_identity empty_column empty_school
""".split()

TEACHER_COLUMNS = """
timestamp empty_name empty_contact empty_column attendance_days
position grades_taught cluster_a school_a cluster_b school_b
first_name last_name gender prior_devcon teaching_experience
subject_areas familiarity_ct familiarity_flowcharts
familiarity_block_coding familiarity_text_coding familiarity_matatag
ct_definition project_facilitation usage_hour_of_code usage_scratch
usage_microbit usage_arduino usage_ai usage_climate_kit
after_summit_conf_ct after_summit_conf_flowcharts
after_summit_conf_coding after_summit_conf_lesson_design
after_summit_conf_matatag after_summit_conf_assessment
after_summit_conf_climate after_summit_conf_inclusion
after_summit_conf_collaboration ct_value_belief
environment_tech_belief matatag_opportunities integration_intention
teaching_identity frequency_ct frequency_flowcharts frequency_coding
frequency_pbl frequency_real_world teaching_challenges
climate_familiarity student_local_issues tech_environment_uses
learning_goals teaching_concerns accommodation_needs privacy_consent
""".split()

assert len(students_raw.columns) == len(STUDENT_COLUMNS), (
    "Student column count changed. Check the source headers."
)

assert len(teachers_raw.columns) == len(TEACHER_COLUMNS), (
    "Teacher column count changed. Check the source headers."
)

student_questions = dict(zip(STUDENT_COLUMNS, students_raw.columns))
teacher_questions = dict(zip(TEACHER_COLUMNS, teachers_raw.columns))

students = students_raw.copy()
teachers = teachers_raw.copy()

students.columns = STUDENT_COLUMNS
teachers.columns = TEACHER_COLUMNS

display(pd.DataFrame({
    "short_name": STUDENT_COLUMNS,
    "original_question": students_raw.columns,
}))

display(pd.DataFrame({
    "short_name": TEACHER_COLUMNS,
    "original_question": teachers_raw.columns,
}))

,short_name,original_question
0,timestamp,Timestamp
1,grade_level,Grade Level
2,first_name,First Name
3,last_name,Last Name
4,gender,Gender
5,age,Age
6,prior_devcon,Have you attended a DEVCON Kids event before?
7,climate_familiarity,How familiar are you with climate change?
8,climate_tech_uses,Which of the following are ways technology can...
9,heard_of_ct,"Before today, have you heard of ""Computational..."


,short_name,original_question
0,timestamp,Timestamp
1,empty_name,Full Name
2,empty_contact,Contact Number
3,empty_column,Column 3
4,attendance_days,Which day(s) will you attend the Climate Champ...
5,position,Position
6,grades_taught,Grade Level(s) You Teach
7,cluster_a,School Cluster
8,school_a,School Name
9,cluster_b,School Cluster 2


## 5. Clean Text and Remove Empty Rows and Columns

Trim surrounding spaces from text and convert empty strings into
missing values.

Remove rows and columns that are completely empty. Keep partially
answered responses.

Assign each submission a response ID and record its original Excel row.
A response ID identifies a submission, not necessarily a unique person.

In [5]:
def clean_text(value):
    if isinstance(value, str):
        value = value.strip()
        return pd.NA if value == "" else value
    return value


def basic_clean(df, prefix):
    result = df.copy()

    # The first response appears on Excel row 2.
    source_rows = pd.Series(
        range(2, len(result) + 2),
        index=result.index,
    )

    for column in result.columns:
        result[column] = result[column].map(clean_text)

    empty_rows = result.isna().all(axis=1)
    result = result.loc[~empty_rows].copy()

    removed_columns = result.columns[
        result.isna().all(axis=0)
    ].tolist()

    result = result.drop(columns=removed_columns)

    result.insert(
        0,
        "response_id",
        [
            f"{prefix}{row - 1:04d}"
            for row in source_rows.loc[result.index]
        ],
    )

    result.insert(
        1,
        "source_excel_row",
        source_rows.loc[result.index].to_numpy(),
    )

    return result, removed_columns, int(empty_rows.sum())


students, student_removed_columns, student_removed_rows = basic_clean(
    students, "S"
)

teachers, teacher_removed_columns, teacher_removed_rows = basic_clean(
    teachers, "T"
)

print("Student empty rows removed:", student_removed_rows)
print("Student empty columns removed:", student_removed_columns)

print("Teacher empty rows removed:", teacher_removed_rows)
print("Teacher empty columns removed:", teacher_removed_columns)

Student empty rows removed: 0
Student empty columns removed: ['attendance_days', 'empty_ct', 'empty_column', 'empty_school']
Teacher empty rows removed: 0
Teacher empty columns removed: ['empty_name', 'empty_contact', 'empty_column']


## 6. Convert Timestamps and Combine School Information

Convert timestamps into datetime values.

Combine the alternative school-name and school-cluster fields into one
school name and one cluster per response.

If both branches contain different values, flag the conflict and leave
the combined field missing until verified. Do not guess missing schools.

In [7]:
def prepare_school_information(df):
    result = df.copy()

    original_timestamps = result["timestamp"].copy()

    result["timestamp"] = pd.to_datetime(
        original_timestamps,
        errors="coerce",
    )

    invalid_dates = (
        original_timestamps.notna()
        & result["timestamp"].isna()
    )

    if invalid_dates.any():
        raise ValueError(
            "Some timestamps could not be converted. "
            "Review the source timestamps."
        )

    school_conflict = (
    result["school_a"].notna()
    & result["school_b"].notna()
    & result["school_a"].astype("string").ne(
        result["school_b"].astype("string")
    )
).fillna(False)

    cluster_a = pd.to_numeric(
    result["cluster_a"], errors="raise"
).astype("Int64")

    cluster_b = pd.to_numeric(
    result["cluster_b"], errors="raise"
).astype("Int64")

    cluster_conflict = (
    cluster_a.notna()
    & cluster_b.notna()
    & cluster_a.ne(cluster_b)
).fillna(False)

    result["school_conflict"] = school_conflict | cluster_conflict

    result["school_name"] = result["school_a"].combine_first(
        result["school_b"]
    )

    combined_cluster = result["cluster_a"].combine_first(
        result["cluster_b"]
    )

    result["school_cluster"] = pd.to_numeric(
        combined_cluster,
        errors="raise",
    ).astype("Int64")

    result.loc[school_conflict, "school_name"] = pd.NA
    result.loc[cluster_conflict, "school_cluster"] = pd.NA

    result["missing_school"] = (
        result["school_name"].isna()
        | result["school_cluster"].isna()
    )

    # Retain the original branch fields if conflicts need review.
    if not result["school_conflict"].any():
        result = result.drop(columns=[
            "school_a",
            "school_b",
            "cluster_a",
            "cluster_b",
        ])

    return result


students = prepare_school_information(students)
teachers = prepare_school_information(teachers)

students["school_level"] = students["grade_level"].map({
    "Grade 5": "Elementary",
    "Grade 6": "Elementary",
    "Grade 7": "High school",
    "Grade 8": "High school",
    "Grade 9": "High school",
})

for label, df in [("Students", students), ("Teachers", teachers)]:
    print(label)
    print(df[["school_name", "school_cluster"]].isna().sum())
    print()

Students
school_name       0
school_cluster    0
dtype: int64

Teachers
school_name       3
school_cluster    3
dtype: int64



## 7. Flag Possible Duplicate Submissions

Compare first and last names after normalizing capitalization and
spacing.

Matching names are treated as possible duplicates, not confirmed
duplicates. Retain these submissions until their school, timestamps,
and answers have been reviewed.

In [8]:
def normalize_name(series):
    return (
        series.astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.casefold()
    )


def flag_duplicates(df):
    result = df.copy()

    name_key = (
        normalize_name(result["first_name"])
        + "|"
        + normalize_name(result["last_name"])
    )

    result["possible_duplicate"] = (
        name_key.notna()
        & name_key.duplicated(keep=False)
    )

    return result


students = flag_duplicates(students)
teachers = flag_duplicates(teachers)

duplicate_columns = [
    "response_id",
    "timestamp",
    "first_name",
    "last_name",
    "school_name",
]

display(
    students.loc[students["possible_duplicate"], duplicate_columns]
    .sort_values(["last_name", "first_name", "timestamp"])
)

display(
    teachers.loc[teachers["possible_duplicate"], duplicate_columns]
    .sort_values(["last_name", "first_name", "timestamp"])
)

,response_id,timestamp,first_name,last_name,school_name
10,S0011,2026-08-13 15:06:43.258,Kesha Chenelle,Balaguer,Sto. Niño Elementary School
14,S0015,2026-08-13 20:01:21.024,Kesha Chenelle,Balaguer,Sto. Niño Elementary School
209,S0210,2026-08-12 15:59:26.670,Jillian,Cases,Dela Paz High School
215,S0216,2026-08-12 20:59:07.940,Jillian,Cases,Dela Paz High School
3,S0004,2026-08-12 20:24:29.459,Mary Samantha,Enriquez,Dela Paz Elementary School
4,S0005,2026-08-12 20:31:52.479,Mary Samantha,Enriquez,Dela Paz Elementary School
135,S0136,2026-08-13 23:22:08.238,Zach Eitan,Galzote,Dela Paz High School
136,S0137,2026-08-13 23:25:47.566,Zach Eitan,Galzote,Dela Paz High School
146,S0147,2026-08-10 17:32:11.866,Prince Mavie,Ilagan,Sagad High School
149,S0150,2026-08-10 17:57:37.349,Prince Mavie,Ilagan,Sagad High School


,response_id,timestamp,first_name,last_name,school_name
34,T0035,2026-08-11 06:59:30.251,LEAH,PADILLA,Rizal High School
36,T0037,2026-08-11 10:53:46.837,LEAH,PADILLA,Rizal High School


## 8. Prepare Student Familiarity Scores

Extract the numeric 1–5 prefix from the student computational-thinking
familiarity answers.

Keep the original labeled answer alongside the numeric score. Check
that every nonblank answer was converted successfully.

Teacher text scales remain labeled because their scoring should be
confirmed against the questionnaire before numeric conversion.

In [9]:
students["ct_familiarity_score"] = (
    students["ct_familiarity"]
    .astype("string")
    .str.extract(r"^\s*([1-5])\s*-", expand=False)
    .astype("Int64")
)

failed_conversion = (
    students["ct_familiarity"].notna()
    & students["ct_familiarity_score"].isna()
)

display(
    students[["ct_familiarity", "ct_familiarity_score"]].head()
)

display(
    students.loc[
        failed_conversion,
        ["response_id", "ct_familiarity"],
    ]
)

assert not failed_conversion.any(), (
    "Some familiarity answers could not be converted."
)

,ct_familiarity,ct_familiarity_score
0,1 - I've never heard of it,1
1,1 - I've never heard of it,1
2,2 - I've heard the term but don't know what it...,2
3,2 - I've heard the term but don't know what it...,2
4,1 - I've never heard of it,1


,response_id,ct_familiarity


## 9. Clean Student Technology Selections

Split the student technology answers into individual selections.
Trim spaces, remove empty items, and remove repeated selections within
the same answer.

Preserve the original response and create a cleaned text version.

Flag answers that select both a technology and "None of the above".
Create numeric indicators for common technologies:

- 1: selected
- 0: not selected
- Blank: missing or contradictory response

Other technology selections remain available in the cleaned text field.

In [10]:
def split_technologies(value):
    if pd.isna(value):
        return []

    selections = [
        item.strip()
        for item in str(value).split(",")
        if item.strip()
    ]

    return list(dict.fromkeys(selections))


technology_lists = students["technologies_used"].map(
    split_technologies
)

students["technologies_cleaned"] = technology_lists.map(
    lambda values: "; ".join(values) if values else pd.NA
)

students["technology_conflict"] = technology_lists.map(
    lambda values: (
        "None of the above" in values
        and len(values) > 1
    )
)

technology_options = {
    "Artificial Intelligence (AI)": "selected_ai",
    "Scratch": "selected_scratch",
    "Python": "selected_python",
    "Micro:bit": "selected_microbit",
    "Arduino": "selected_arduino",
    "Climate Action Kit": "selected_climate_kit",
}

for option, column in technology_options.items():
    indicator_values = []

    for index, selections in technology_lists.items():
        if (
            not selections
            or students.loc[index, "technology_conflict"]
        ):
            indicator_values.append(pd.NA)
        else:
            indicator_values.append(int(option in selections))

    students[column] = pd.array(
        indicator_values,
        dtype="Int64",
    )

display(
    students.loc[
        students["technology_conflict"],
        [
            "response_id",
            "technologies_used",
            "technologies_cleaned",
        ],
    ]
)

,response_id,technologies_used,technologies_cleaned
124,S0125,"Artificial Intelligence (AI), None of the above",Artificial Intelligence (AI); None of the above
130,S0131,"Artificial Intelligence (AI), None of the above",Artificial Intelligence (AI); None of the above
180,S0181,"Artificial Intelligence (AI), None of the above",Artificial Intelligence (AI); None of the above


## 10. Inspect Missing Answers and Arrange Columns

Place submission identifiers, timestamps, and school information at
the beginning of each dataset.

Summarize missing answers without filling them with invented values
or removing partially answered submissions.

In [11]:
def arrange_columns(df):
    first_columns = [
        "response_id",
        "source_excel_row",
        "timestamp",
        "first_name",
        "last_name",
        "school_name",
        "school_cluster",
    ]

    return df[
        first_columns
        + [
            column
            for column in df.columns
            if column not in first_columns
        ]
    ].copy()


def missing_summary(df):
    return pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": (
            df.isna().mean().mul(100).round(1)
        ),
    }).sort_values("missing_count", ascending=False)


students = arrange_columns(students)
teachers = arrange_columns(teachers)

display(missing_summary(students))
display(missing_summary(teachers))

,missing_count,missing_percent
accommodation_needs,90,37.3
selected_climate_kit,3,1.2
selected_scratch,3,1.2
selected_ai,3,1.2
selected_microbit,3,1.2
selected_python,3,1.2
selected_arduino,3,1.2
first_name,0,0.0
gender,0,0.0
age,0,0.0


,missing_count,missing_percent
accommodation_needs,12,21.8
school_name,3,5.5
school_cluster,3,5.5
source_excel_row,0,0.0
response_id,0,0.0
last_name,0,0.0
first_name,0,0.0
attendance_days,0,0.0
timestamp,0,0.0
grades_taught,0,0.0


## 11. Create Review Tables and Documentation

Prepare three supporting sheets:

- **Review:** flagged records and spaces for review decisions.
- **Dictionary:** short column names and original questions.
- **Cleaning Notes:** transformations, policies, and limitations.

Review decisions are recorded manually. Changing a dropdown does not
automatically remove or correct a response in Cleaned Data.

In [12]:
def build_review(df):
    records = []

    checks = {
        "possible_duplicate": (
            "Possible duplicate",
            "Compare school, answers, and timestamps before deciding "
            "which submission to retain.",
        ),
        "missing_school": (
            "Missing school",
            "Verify the school and cluster. Otherwise retain the blanks.",
        ),
        "school_conflict": (
            "School conflict",
            "The school branches disagree. Check the original values.",
        ),
        "technology_conflict": (
            "Conflicting technology selections",
            "A technology and None of the above were both selected. "
            "Verify the intended answer.",
        ),
    }

    columns = [
        "response_id",
        "source_excel_row",
        "first_name",
        "last_name",
        "school_name",
        "issue",
        "suggested_action",
        "decision",
        "review_notes",
    ]

    for flag, (issue, action) in checks.items():
        if flag not in df.columns:
            continue

        for _, row in df.loc[df[flag].fillna(False)].iterrows():
            records.append({
                "response_id": row["response_id"],
                "source_excel_row": row["source_excel_row"],
                "first_name": row["first_name"],
                "last_name": row["last_name"],
                "school_name": row["school_name"],
                "issue": issue,
                "suggested_action": action,
                "decision": "Pending",
                "review_notes": "",
            })

    return pd.DataFrame(records, columns=columns)


def build_dictionary(df, questions):
    derived = {
        "response_id": "Submission ID, not a unique person ID.",
        "source_excel_row": "Original Excel response-sheet row number.",
        "school_name": "Combined school_a and school_b.",
        "school_cluster": "Combined cluster_a and cluster_b.",
        "school_level": "Derived from student grade level.",
        "possible_duplicate": "True when normalized names repeat.",
        "missing_school": "True when school name or cluster is missing.",
        "school_conflict": "True when school branches disagree.",
        "ct_familiarity_score": "Numeric 1–5 prefix from ct_familiarity.",
        "technologies_cleaned": (
            "Trimmed unique technology selections separated by semicolons."
        ),
        "technology_conflict": (
            "True when None of the above appears with another technology."
        ),
    }

    rows = []

    for column in df.columns:
        description = questions.get(
            column,
            derived.get(column, ""),
        )

        if column.startswith("selected_"):
            description = (
                "1 = selected; 0 = not selected; "
                "blank = missing or contradictory technology answer."
            )

        description = re.sub(
            r"\s+",
            " ",
            str(description).replace("_x000a_", " "),
        ).strip()

        rows.append({
            "column_name": column,
            "original_question_or_derivation": description,
            "data_type": str(df[column].dtype),
            "missing_count": int(df[column].isna().sum()),
        })

    return pd.DataFrame(rows)


def build_notes(group, df, removed_columns, removed_rows, questions):
    rows = [
        ("Source", f"{group}_pre_event.xlsx / Form Responses 1"),
        ("Rows retained", str(len(df))),
        ("Empty rows removed", str(removed_rows)),
        (
            "Possible duplicate rows",
            str(int(df["possible_duplicate"].sum())),
        ),
        (
            "Missing school rows",
            str(int(df["missing_school"].sum())),
        ),
        (
            "Text cleaning",
            "Trimmed surrounding spaces. Empty strings became missing. "
            "Explicit None and NA answers remain text.",
        ),
        (
            "School consolidation",
            "Combined alternative school fields. Missing values were not guessed.",
        ),
        (
            "Duplicate policy",
            "Possible duplicates retained pending review.",
        ),
        (
            "Missing-answer policy",
            "Missing answers remain blank and were not replaced with zero.",
        ),
        (
            "Removed empty columns",
            "; ".join(
                re.sub(r"\s+", " ", questions[column]).strip()
                for column in removed_columns
            ),
        ),
        (
            "Review workflow",
            "Record decisions in Review, then update the data deliberately. "
            "Dropdown choices do not automatically change Cleaned Data.",
        ),
        (
            "Source trace",
            "response_id identifies a submission. source_excel_row points "
            "to the original Excel row.",
        ),
        (
            "Personal information",
            "Names and accommodation answers remain in this workbook. "
            "Exclude identifying information from shared analysis results.",
        ),
        (
            "Other source sheets",
            "Not appended because they may contain overlapping responses.",
        ),
    ]

    if group == "students":
        rows.extend([
            (
                "Technology conflict rows",
                str(int(df["technology_conflict"].sum())),
            ),
            (
                "Technology indicators",
                "1 = selected; 0 = not selected. Blank for missing or "
                "contradictory responses. Other selections remain in "
                "technologies_cleaned.",
            ),
        ])
    else:
        rows.append((
            "Confidence question wording",
            "Nine questions in the source main sheet say After completing "
            "the summit. Confirm their timing before treating them as "
            "pre-event confidence.",
        ))

    return pd.DataFrame(rows, columns=["item", "details"])


student_review = build_review(students)
teacher_review = build_review(teachers)

student_dictionary = build_dictionary(students, student_questions)
teacher_dictionary = build_dictionary(teachers, teacher_questions)

student_notes = build_notes(
    "students",
    students,
    student_removed_columns,
    student_removed_rows,
    student_questions,
)

teacher_notes = build_notes(
    "teachers",
    teachers,
    teacher_removed_columns,
    teacher_removed_rows,
    teacher_questions,
)

print("Student review entries:", len(student_review))
print("Teacher review entries:", len(teacher_review))

Student review entries: 23
Teacher review entries: 5


## 12. Export Readable Excel Workbooks

Save separate student and teacher workbooks in `data/cleaned/`.

Each workbook includes Cleaned Data, Review, Dictionary, and Cleaning
Notes sheets.

Apply purple headers, alternating row colors, filters, frozen panes,
wrapped text, formatted timestamps, and review-decision dropdowns.

Rerunning this step replaces the output workbooks, including any manual
review notes entered in them. Close the files in Excel before exporting.

In [13]:
def format_sheet(ws):
    ws.sheet_view.showGridLines = False

    ws.freeze_panes = (
        "C2"
        if ws.title in ["Cleaned Data", "Review"]
        else "A2"
    )

    purple = PatternFill("solid", fgColor="4C286B")
    pale_purple = PatternFill("solid", fgColor="F3EFF8")
    pale_yellow = PatternFill("solid", fgColor="FFF2CC")

    widths = {}

    for column_index, header_cell in enumerate(ws[1], start=1):
        header = str(header_cell.value or "")

        if ws.title == "Dictionary":
            width = [34, 100, 20, 18][column_index - 1]

        elif ws.title == "Cleaning Notes":
            width = [32, 110][column_index - 1]

        elif ws.title == "Review":
            width = [
                18, 18, 24, 24, 42, 34, 75, 24, 55
            ][column_index - 1]

        elif header == "timestamp":
            width = 24

        elif header == "school_name":
            width = 42

        elif any(
            word in header
            for word in [
                "definition",
                "learning_goals",
                "climate_problem",
                "concerns",
                "challenges",
                "technologies",
                "privacy_consent",
                "accommodation",
            ]
        ):
            width = 55

        elif "familiarity" in header or "after_summit" in header:
            width = 32

        else:
            width = 24

        widths[column_index] = width

        column_letter = get_column_letter(column_index)
        ws.column_dimensions[column_letter].width = width

        header_cell.fill = purple
        header_cell.font = Font(
            name="Calibri",
            size=11,
            bold=True,
            color="FFFFFF",
        )
        header_cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True,
        )

    ws.row_dimensions[1].height = 44

    for row in ws.iter_rows(min_row=2):
        maximum_lines = 1

        for cell in row:
            cell.font = Font(name="Calibri", size=11)
            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True,
            )

            if cell.row % 2 == 0:
                cell.fill = pale_purple

            # Keep responses beginning with "=" as literal text.
            if isinstance(cell.value, str) and cell.value.startswith("="):
                cell.data_type = "s"

            if isinstance(cell.value, datetime):
                cell.number_format = "yyyy-mm-dd hh:mm:ss"
                display_value = cell.value.strftime(
                    "%Y-%m-%d %H:%M:%S"
                )
            else:
                display_value = (
                    "" if cell.value is None else str(cell.value)
                )

            characters_per_line = max(
                10,
                int(widths[cell.column]) - 3,
            )

            estimated_lines = sum(
                max(1, ceil(len(line) / characters_per_line))
                for line in display_value.split("\n")
            )

            maximum_lines = max(maximum_lines, estimated_lines)

        ws.row_dimensions[row[0].row].height = min(
            409,
            max(28, maximum_lines * 15 + 8),
        )

    if ws.max_row > 1:
        table_name = (
            re.sub(r"[^A-Za-z0-9_]", "", ws.title) + "Table"
        )

        table = Table(
            displayName=table_name,
            ref=(
                f"A1:"
                f"{get_column_letter(ws.max_column)}{ws.max_row}"
            ),
        )

        table.tableStyleInfo = TableStyleInfo(
            name="TableStyleMedium4",
            showFirstColumn=False,
            showLastColumn=False,
            showRowStripes=False,
            showColumnStripes=False,
        )

        ws.add_table(table)

    if ws.title == "Review" and ws.max_row > 1:
        dropdown = DataValidation(
            type="list",
            formula1=(
                '"Pending,Verified - retain,Verified - exclude,Corrected"'
            ),
            allow_blank=False,
        )

        ws.add_data_validation(dropdown)
        dropdown.add(f"H2:H{ws.max_row}")

        for row in ws.iter_rows(
            min_row=2,
            min_col=8,
            max_col=9,
        ):
            for cell in row:
                cell.fill = pale_yellow


def export_workbook(path, data, review, dictionary, notes):
    with pd.ExcelWriter(
        path,
        engine="openpyxl",
        datetime_format="yyyy-mm-dd hh:mm:ss",
    ) as writer:

        sheets = {
            "Cleaned Data": data,
            "Review": review,
            "Dictionary": dictionary,
            "Cleaning Notes": notes,
        }

        for sheet_name, dataframe in sheets.items():
            dataframe.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False,
            )

        for worksheet in writer.book.worksheets:
            format_sheet(worksheet)

    print("Saved:", path)


student_output = CLEANED / "students_pre_event_cleaned.xlsx"
teacher_output = CLEANED / "teachers_pre_event_cleaned.xlsx"

export_workbook(
    student_output,
    students,
    student_review,
    student_dictionary,
    student_notes,
)

export_workbook(
    teacher_output,
    teachers,
    teacher_review,
    teacher_dictionary,
    teacher_notes,
)

Saved: C:\Users\User\Desktop\DEVCON-DATA-ANALYTICS\SEPTEMBER _6_DATA\data\cleaned\students_pre_event_cleaned.xlsx
Saved: C:\Users\User\Desktop\DEVCON-DATA-ANALYTICS\SEPTEMBER _6_DATA\data\cleaned\teachers_pre_event_cleaned.xlsx


## 13. Verify Saved Files

Reload the exported Cleaned Data sheets and check that row counts,
column names, and unique response IDs were preserved.

Display the number of retained submissions and outstanding review flags.
Successful export does not resolve the flagged data-quality issues.

In [14]:
summary = pd.DataFrame({
    "dataset": ["Students", "Teachers"],
    "rows_retained": [
        len(students),
        len(teachers),
    ],
    "possible_duplicate_rows": [
        int(students["possible_duplicate"].sum()),
        int(teachers["possible_duplicate"].sum()),
    ],
    "missing_school_rows": [
        int(students["missing_school"].sum()),
        int(teachers["missing_school"].sum()),
    ],
    "technology_conflict_rows": [
        int(students["technology_conflict"].sum()),
        pd.NA,
    ],
})

display(summary)

for path, expected in [
    (student_output, students),
    (teacher_output, teachers),
]:
    saved = pd.read_excel(
        path,
        sheet_name="Cleaned Data",
        keep_default_na=False,
        na_values=[""],
    )

    assert len(saved) == len(expected)
    assert saved["response_id"].is_unique
    assert saved.columns.tolist() == expected.columns.tolist()

print("Both cleaned workbooks were saved and checked successfully.")

,dataset,rows_retained,possible_duplicate_rows,missing_school_rows,technology_conflict_rows
0,Students,241,20,0,3
1,Teachers,55,2,3,<NA>


Both cleaned workbooks were saved and checked successfully.


## Initial Cleaning Conclusion

The datasets now have shorter column names, standardized blank values,
combined school information, submission identifiers, and documented
review flags.

Student computational-thinking familiarity scores and technology
indicators were prepared while preserving the original answers.

Possible duplicate submissions, missing teacher school information, and
contradictory technology answers remain subject to review. These records
were not automatically deleted or corrected.

The exported workbooks can be loaded into the analysis notebooks.
Outstanding review decisions should be resolved or explicitly accounted
for before reporting final statistics.